In [1]:
import warnings
for warn in [UserWarning, FutureWarning]: warnings.filterwarnings("ignore", category = warn)

import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import hub
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import torchvision.models as models

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, recall_score, precision_score, balanced_accuracy_score, accuracy_score, classification_report
from sklearn.utils import shuffle

import scipy

from tqdm import tqdm

from datasets import load_dataset, Dataset, Audio
import librosa
from transformers import AutoFeatureExtractor, Wav2Vec2ForSequenceClassification

# from models.swishnet import SwishNet
from models.cnn import CNNModel

from src.utils import AphasiaDatasetMFCC, AphasiaDatasetSpectrogram, AphasiaDatasetWaveform

from collections import Counter

# from torch_audiomentations import Compose, AddColoredNoise, PitchShift, Gain

/home/zakhar/miniconda3/envs/aphasia_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1787324111.559226   75748 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787324111.801921   75748 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787324112.929163   75748 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to 

In [2]:
SEED = 1984

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

gen = set_seed(SEED)

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cuda time!!!


In [4]:
DATA_DIR = os.path.join(os.getcwd(), 'data')
VOICES_DIR = os.path.join(DATA_DIR, 'Voices')
APHASIA_DIR = os.path.join(VOICES_DIR, 'Aphasia')
NORM_DIR = os.path.join(VOICES_DIR, 'Norm')

In [5]:
train_dataset = AphasiaDatasetSpectrogram(os.path.join(DATA_DIR, "train_filenames.csv"), root_dir=VOICES_DIR, seed=SEED)
val_dataset = AphasiaDatasetSpectrogram(os.path.join(DATA_DIR, "val_filenames.csv"), root_dir=VOICES_DIR, seed=SEED)
test_dataset = AphasiaDatasetSpectrogram(os.path.join(DATA_DIR, "test_filenames.csv"), root_dir=VOICES_DIR, seed=SEED)

# Балансировка классов для train
train_labels = [label for _, label in train_dataset.data]
class_counts = Counter(train_labels)
if len(class_counts) < 2:
    raise ValueError("Один из классов отсутствует в тренировочном наборе")

class_weights = {label: 1.0 / count for label, count in class_counts.items()}
weights = [class_weights[label] for _, label in train_dataset.data]
gen = set_seed(SEED)
train_sampler = WeightedRandomSampler(weights, num_samples=len(train_dataset), replacement=True, generator=gen)

In [6]:
def pad_sequence(batch):
    """Pad spectrograms in batch to same length"""
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    spectrograms, labels = zip(*batch)
    
    # Find maximum sequence length in batch
    max_len = max(s.shape[1] for s in spectrograms)
    freq_bins = spectrograms[0].shape[2]

    # Create padded tensor
    padded = torch.zeros(len(spectrograms), 1, max_len, freq_bins)
    for i, s in enumerate(spectrograms):
        padded[i, :, :s.shape[1], :] = s

    return padded, torch.stack(labels)

In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=8, sampler=train_sampler, collate_fn=pad_sequence, drop_last=True, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=pad_sequence, drop_last=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=pad_sequence, drop_last=True, num_workers=0)

In [8]:
def train_model(model, dl_train, dl_val, epochs=1, lr=0.001, device="cpu", path="checkpoints", model_name="model", seed=SEED):
    set_seed(seed)
    if getattr(dl_train, "sampler", None) is not None:
        sampler_gen = torch.Generator()
        sampler_gen.manual_seed(seed)
        dl_train.sampler.generator = sampler_gen

    model = model.to(device)
    criterion = nn.CrossEntropyLoss().to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
        
    train_loss_list = []
    val_loss_list = []
    train_acc_list = []
    val_acc_list = []
    max_val_acc = 0
    for epoch in tqdm(range(epochs), desc="Training model"):
        model.train()
        total_train_loss = 0
        total_val_loss = 0
        
        train_acc = []
        val_acc = []
        for features, target in dl_train:
            features, target = features.to(device), target.to(device)
            optimizer.zero_grad()
            
            output = model(features).squeeze()

            preds = torch.argmax(output, dim=1).cpu().detach().numpy()
            train_acc.append(accuracy_score(target.cpu().detach().numpy(), preds))
            
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.detach().item()
        
        avg_train_acc = np.stack(train_acc, axis=0).mean()

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_loss_list.append(avg_train_loss)
        train_acc_list.append(avg_train_acc)
                
        model.eval()
        
        with torch.no_grad():
            for features, target in dl_val:
                features, target = features.to(device), target.to(device)
                
                output = model(features).squeeze()
                
                preds = torch.argmax(output, dim=1).cpu().detach().numpy()
                val_acc.append(accuracy_score(target.cpu().detach().numpy(), preds))
                loss = criterion(output, target)
                total_val_loss += loss.detach().item()
        
        avg_val_acc = np.stack(val_acc, axis=0).mean()
        avg_val_loss = total_val_loss / len(dl_val)
        val_loss_list.append(avg_val_loss)
        val_acc_list.append(avg_val_acc)
        
        if scheduler:
            try:
                scheduler.step()
            except:
                scheduler.step(avg_val_loss)
        
        if max_val_acc <= avg_val_acc:
            if not os.path.exists(path):
                os.makedirs(path)
                
            chkp = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                "plots": {
                    "train_loss": train_loss_list,
                    "train_acc": train_acc_list,
                    "val_loss": val_loss_list,
                    "val_acc": val_acc_list
                },
            }
            torch.save(chkp, os.path.join(path, model_name + ".pt"))
            max_val_acc = avg_val_acc

        # if epoch % 10 == 0:
        tqdm.write(f"Epoch {epoch}: train loss: {avg_train_loss:.3f}, train balanced acc: {avg_train_acc:.2f}, test loss: {avg_val_loss:.3f}, test balanced acc: {avg_val_acc:.2f}, lr: {optimizer.param_groups[0]['lr']}")

            
    return model, train_loss_list, val_loss_list, train_acc_list, val_acc_list

In [9]:
from models.cnn import CNNModel

set_seed(SEED)
cnn = CNNModel(num_classes=2, in_channels=1)

In [10]:
cnn, train_l, val_l, train_accuracy, val_accuracy = train_model(cnn, train_dataloader, val_dataloader, epochs=30, lr=1e-3, 
                                                                device=DEVICE, path=os.path.join("checkpoints", "cnn"), model_name="cnn_model_spec")

Training model:   3%|▎         | 1/30 [02:01<58:47, 121.62s/it]

Epoch 0: train loss: 0.589, train balanced acc: 0.68, test loss: 0.729, test balanced acc: 0.54, lr: 0.0009755527298894294


Training model:   7%|▋         | 2/30 [04:04<57:05, 122.34s/it]

Epoch 1: train loss: 0.562, train balanced acc: 0.71, test loss: 0.575, test balanced acc: 0.73, lr: 0.0009046039886902864


Training model:  10%|█         | 3/30 [06:13<56:26, 125.41s/it]

...

Epoch 27: train loss: 0.200, train balanced acc: 0.92, test loss: 0.679, test balanced acc: 0.78, lr: 9.639601130971425e-05


Training model:  97%|█████████▋| 29/30 [1:03:32<02:11, 131.44s/it]

Epoch 28: train loss: 0.183, train balanced acc: 0.93, test loss: 0.781, test balanced acc: 0.83, lr: 2.5447270110570967e-05


Training model: 100%|██████████| 30/30 [1:05:42<00:00, 131.43s/it]

Epoch 29: train loss: 0.174, train balanced acc: 0.93, test loss: 0.792, test balanced acc: 0.83, lr: 1e-06


In [11]:
# import matplotlib.pyplot as plt

# # Plotting training and validation metrics
# plt.figure(figsize=(15, 10))

# # Plot 1: Training Loss
# plt.subplot(2, 2, 1)
# plt.plot(30, train_l, 'b-', label='Train Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss')
# plt.legend()
# plt.grid(True)

# # Plot 2: Validation Accuracy
# plt.subplot(2, 2, 2)
# plt.plot(30, val_accuracy, 'r-', label='Validation Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.title('Validation Accuracy')
# plt.legend()
# plt.grid(True)

# # Plot 4: All metrics together
# plt.subplot(2, 1, 2)
# plt.plot(30, train_l, 'b-', label='Train Loss', alpha=0.7)
# plt.plot(30, val_accuracy, 'r-', label='Val Accuracy', alpha=0.7)
# plt.xlabel('Epoch')
# plt.ylabel('Scores')
# plt.title('All Metrics')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

In [12]:
state_d = torch.load(os.path.join("checkpoints", "cnn", "cnn_model_spec.pt"), weights_only=False)

cnn = CNNModel(num_classes=2, in_channels=1)
cnn.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [13]:
test_data = pd.read_csv(os.path.join(DATA_DIR, 'test_filenames.csv'))
DATA_DIR = os.path.join(os.getcwd(), 'data')
VOICES_DIR = os.path.join(DATA_DIR, 'Voices_wav')
APHASIA_DIR = os.path.join(VOICES_DIR, 'Aphasia')
NORM_DIR = os.path.join(VOICES_DIR, 'Norm')

In [14]:
def pad_sequence(batch):
    """Pad spectrograms in batch to same length"""
    if not batch:
        return torch.zeros(0), torch.zeros(0)

    spectrograms, labels = zip(*batch)
    
    # Find maximum sequence length in batch
    max_len = max(s.shape[1] for s in spectrograms)
    freq_bins = spectrograms[0].shape[2]

    # Create padded tensor
    padded = torch.zeros(len(spectrograms), 1, max_len, freq_bins)
    for i, s in enumerate(spectrograms):
        padded[i, :, :s.shape[1], :] = s

    return padded, torch.stack(labels)

def test_model_for_each_participant(model, test_data):
    model = model.to("cpu")
        
    model.eval()

    test_data["ID"] = test_data["file_name"].apply(
        lambda x: str(x).split("-")[0] + str(x).split("-")[1])
    test_data.head()
    IDs = test_data["ID"].unique()

    all_preds = []
    all_labels = []
    
    errors = []
    
    with torch.no_grad():
        for participant_id in tqdm(IDs):
            participant_samples = test_data[test_data["ID"] == participant_id]
            preds = []
            for ind, participant_sample in participant_samples.iterrows():

                sgnl_path = participant_sample["file_name"]

                if sgnl_path[0] == "N":
                    sgnl_path = os.path.join(NORM_DIR, sgnl_path)
                else:
                    sgnl_path = os.path.join(APHASIA_DIR, sgnl_path)
                    

                chunks = train_dataset.process_audio(sgnl_path)
                max_len = max(s.shape[1] for s in chunks)
                freq_bins = chunks[0].shape[2]

                # Create padded tensor
                padded = torch.zeros(len(chunks), 1, max_len, freq_bins, dtype=torch.float32)
                
                # padded = torch.zeros(len(chunks), MFCC, SEQUENCE_LENGTH)
                for i, s in enumerate(chunks):
                    padded[i, :, :s.shape[1], :] = s
                # print(model(torch.from_numpy(np.array(padded)).unsqueeze(1)).detach().numpy().shape)
                # print(padded.shape)
                pred = model(padded.detach())# .argmax(axis=-1)
                seg_probs = torch.softmax(pred, dim=1)
                file_probs = seg_probs.mean(dim=0)  
                
                preds.append(file_probs)
            labels = participant_samples["label"]
            # print(preds)
            # print(preds[0].shape)
            pred = np.stack(preds, axis=0)# .mean(axis=-2).argmax(axis=-1)
            # print(pred.shape)
            pred = pred.mean(axis=-2).argmax(axis=-1)

            all_preds.append(pred)
                
            all_labels.append(labels.values[0])
            errs = participant_samples[labels.values != pred]
            if len(errs) > 0:
                errors.append(errs)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    print(classification_report(all_labels, all_preds))

    
    return all_preds, errors


In [15]:
test_model_for_each_participant(cnn, test_data)

100%|██████████| 72/72 [19:41<00:00, 16.41s/it]

              precision    recall  f1-score   support

           0       0.95      0.78      0.86        23
           1       0.91      0.98      0.94        49

    accuracy                           0.92        72
   macro avg       0.93      0.88      0.90        72
weighted avg       0.92      0.92      0.91        72



(array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1]),
 [               file_name  label     ID
  4  N-0990-RAT-1-bike.wav      0  N0990
  5  N-0990-RAT-1-robb.wav      0  N0990,
                  file_name  label     ID
  16  N-0940-RAT-1-robb.wav      0  N0940
  17  N-0940-RAT-1-bike.wav      0  N0940,
                  file_name  label     ID
  22  N-1005-RAT-1-bike.wav      0  N1005
  23  N-1005-RAT-1-robb.wav      0  N1005,
                  file_name  label     ID
  26  N-0912-RAT-1-robb.wav      0  N0912
  27  N-0912-RAT-1-bike.wav      0  N0912,
                  file_name  label     ID
  30  A-1237-RAT-1-bike.wav      0  A1237
  31  A-1237-RAT-1-robb.wav      0  A1237,
                 file_name  label    ID
  64  A-752-RAT-1-bike.wav      1  A752])

In [16]:
from pydub import AudioSegment

def test_model_no_chunks(model, test_data):
    model = model.to("cpu")
        
    model.eval()

    test_data["ID"] = test_data["file_name"].apply(
        lambda x: str(x).split("-")[0] + str(x).split("-")[1])
    test_data.head()
    IDs = test_data["ID"].unique()

    all_preds_aggr = []
    all_labels_aggr = []
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for participant_id in tqdm(IDs):
            participant_samples = test_data[test_data["ID"] == participant_id]
            preds_aggr = []
            labels_aggr = []
            for ind, participant_sample in participant_samples.iterrows():

                sgnl_path = participant_sample["file_name"]

                if sgnl_path[0] == "N":
                    sgnl_path = os.path.join(NORM_DIR, sgnl_path)
                else:
                    sgnl_path = os.path.join(APHASIA_DIR, sgnl_path)
                    
                # y, sr = librosa.load(sgnl_path, sr=8_000)
                audio = AudioSegment.from_file(sgnl_path, format="3gp")
                y = train_dataset.preprocess(audio)

                pred = model(y[None, ...]).detach().numpy().squeeze().argmax(axis=-1)
                
                labels_aggr.append(participant_sample['label'])
                preds_aggr.append(pred)

            pred = scipy.stats.mode(np.array(preds_aggr))

            all_preds_aggr.append(pred.mode)
            all_labels_aggr.append(labels_aggr[0])
            
            all_preds.extend(preds_aggr)
            all_labels.extend(labels_aggr)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    print("Without aggr")
    print(classification_report(all_labels, all_preds))
    
    print("With aggr")
    print(classification_report(all_labels_aggr, all_preds_aggr))
    
    return all_preds, all_preds_aggr

In [17]:
_, _ = test_model_no_chunks(cnn, test_data)

100%|██████████| 72/72 [17:53<00:00, 14.91s/it]

Without aggr
              precision    recall  f1-score   support

           0       0.97      0.74      0.84        46
           1       0.91      0.99      0.95       122

    accuracy                           0.92       168
   macro avg       0.94      0.87      0.89       168
weighted avg       0.93      0.92      0.92       168

With aggr
              precision    recall  f1-score   support

           0       1.00      0.74      0.85        23
           1       0.89      1.00      0.94        49

    accuracy                           0.92        72
   macro avg       0.95      0.87      0.90        72
weighted avg       0.93      0.92      0.91        72

